# Voice Emotion Recognition — end to end

Run every cell top to bottom. This downloads RAVDESS, extracts features, trains a RandomForest classifier plus a pitch/energy heuristic baseline, evaluates both on 4 held-out actors, and downloads the trained artifacts to your machine so you can run `streamlit_app.py` locally afterward.

Takes about 10-15 minutes total on Colab's default CPU runtime. I used T4 to train my model here.

## 1. Install dependencies

In [1]:
!pip install -q librosa scikit-learn soundfile joblib pandas numpy tqdm

## 2. Download and extract the dataset

[RAVDESS](https://zenodo.org/records/1188976) speech-only audio: 1,440 clips, 24 actors, 8 emotions, CC BY-NC-SA 4.0. Public dataset. ~208MB, downloads straight from Zenodo.

In [2]:
import os

os.makedirs("data/raw", exist_ok=True)

if not os.path.isdir("data/raw/Actor_01"):
    !curl -L -o data/Audio_Speech_Actors_01-24.zip "https://zenodo.org/records/1188976/files/Audio_Speech_Actors_01-24.zip?download=1"
    !unzip -q data/Audio_Speech_Actors_01-24.zip -d data/raw
    !rm data/Audio_Speech_Actors_01-24.zip
else:
    print("data/raw already populated, skipping download.")

import glob
wav_count = len(glob.glob("data/raw/Actor_*/*.wav"))
print(f"{wav_count} wav files ready.")
assert wav_count == 1440, "Expected 1440 files - check the download/unzip above."

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  198M  100  198M    0     0  94.0M      0  0:00:02  0:00:02 --:--:-- 94.0M
1440 wav files ready.


## 3. Feature extraction

RAVDESS filenames self-encode their label, e.g. `03-01-06-01-02-01-12.wav` =
`modality-vocalchannel-EMOTION-intensity-statement-repetition-actor`.

Two feature sets:
- `HEURISTIC_FEATURES`: mean pitch + RMS energy only - what a person could eyeball.
- `MODEL_FEATURES`: 40 MFCCs (mean + std) + pitch + energy + zero-crossing rate - what the trained classifier uses.

Uses `librosa.piptrack` rather than `librosa.pyin` for pitch - pyin is ~35x slower
(8.5 sec/file vs 0.25 sec/file) and precision beyond what a heuristic baseline needs.

In [3]:
from pathlib import Path
import librosa
import numpy as np

EMOTION_MAP = {
    "01": "neutral", "02": "calm", "03": "happy", "04": "sad",
    "05": "angry", "06": "fearful", "07": "disgust", "08": "surprised",
}
SAMPLE_RATE = 22050


def parse_label(wav_path: Path) -> dict:
    parts = wav_path.stem.split("-")
    emotion_code = parts[2]
    actor = int(parts[6])
    return {
        "emotion": EMOTION_MAP[emotion_code],
        "actor": actor,
        "gender": "female" if actor % 2 == 0 else "male",
    }


def extract_features(wav_path: Path) -> dict:
    y, sr = librosa.load(wav_path, sr=SAMPLE_RATE)
    y, _ = librosa.effects.trim(y, top_db=25)

    rms = float(np.mean(librosa.feature.rms(y=y)))

    pitches, magnitudes = librosa.piptrack(y=y, sr=sr, fmin=75, fmax=600)
    idx = magnitudes.argmax(axis=0)
    frame_pitches = pitches[idx, np.arange(pitches.shape[1])]
    voiced_f0 = frame_pitches[frame_pitches > 0]
    mean_pitch = float(np.mean(voiced_f0)) if voiced_f0.size else 0.0
    pitch_std = float(np.std(voiced_f0)) if voiced_f0.size else 0.0

    mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=40)
    mfcc_mean = mfcc.mean(axis=1)
    mfcc_std = mfcc.std(axis=1)

    zcr = float(np.mean(librosa.feature.zero_crossing_rate(y)))

    features = {"rms_energy": rms, "mean_pitch": mean_pitch, "pitch_std": pitch_std, "zcr": zcr}
    for i, v in enumerate(mfcc_mean):
        features[f"mfcc_mean_{i}"] = float(v)
    for i, v in enumerate(mfcc_std):
        features[f"mfcc_std_{i}"] = float(v)
    return features


HEURISTIC_FEATURES = ["mean_pitch", "rms_energy"]
MODEL_FEATURES = (
    ["mean_pitch", "pitch_std", "rms_energy", "zcr"]
    + [f"mfcc_mean_{i}" for i in range(40)]
    + [f"mfcc_std_{i}" for i in range(40)]
)
print("Feature functions defined.")

Feature functions defined.


In [4]:
import pandas as pd
from tqdm.auto import tqdm

DATA_DIR = Path("data/raw")
wav_paths = sorted(DATA_DIR.glob("Actor_*/*.wav"))

rows = []
for wav_path in tqdm(wav_paths, desc="Extracting features"):
    label = parse_label(wav_path)
    feats = extract_features(wav_path)
    rows.append({"file": str(wav_path), **label, **feats})

df = pd.DataFrame(rows)
os.makedirs("data", exist_ok=True)
df.to_csv("data/features.csv", index=False)
print(f"Wrote {len(df)} rows to data/features.csv")
df.head()

Extracting features:   0%|          | 0/1440 [00:00<?, ?it/s]

Wrote 1440 rows to data/features.csv


,file,emotion,actor,gender,rms_energy,mean_pitch,pitch_std,zcr,mfcc_mean_0,mfcc_mean_1,...,mfcc_std_30,mfcc_std_31,mfcc_std_32,mfcc_std_33,mfcc_std_34,mfcc_std_35,mfcc_std_36,mfcc_std_37,mfcc_std_38,mfcc_std_39
0,data/raw/Actor_01/03-01-01-01-01-01-01.wav,neutral,1,male,0.005606,303.060577,163.338959,0.084515,-481.905640,125.548500,...,5.043580,6.008196,4.981233,5.224856,3.505803,4.612251,3.599945,4.756579,4.173026,3.365785
1,data/raw/Actor_01/03-01-01-01-01-02-01.wav,neutral,1,male,0.006037,281.469330,164.735229,0.074784,-478.835449,118.245621,...,6.304369,5.902667,3.916972,5.613611,4.745112,4.280282,3.387980,3.937891,2.880065,4.782964
2,data/raw/Actor_01/03-01-01-01-02-01-01.wav,neutral,1,male,0.007029,306.818420,166.207596,0.094927,-467.461426,123.837830,...,4.268163,4.284394,4.266775,5.544558,3.952373,3.957697,3.921084,4.442653,3.555941,4.286138
3,data/raw/Actor_01/03-01-01-01-02-02-01.wav,neutral,1,male,0.006595,263.131775,167.457886,0.099302,-467.978668,114.493721,...,4.672279,5.068848,6.422096,5.591658,3.740158,4.405075,3.594912,3.586221,3.953244,4.278379
4,data/raw/Actor_01/03-01-02-01-01-01-01.wav,calm,1,male,0.003689,237.602173,156.868073,0.098087,-532.076782,116.151115,...,5.229302,4.045067,4.121466,5.430986,4.421092,5.302922,3.330209,3.954444,3.572410,5.387924


## 4. Train + evaluate

Held out by **actor** (21-24), not by random row - RAVDESS has multiple clips per actor,
so a random row split would leak the same voice into train and test and inflate accuracy.

Two models are trained for comparison:
- **Heuristic**: nearest-centroid classifier using only mean pitch + RMS energy.
- **Model**: RandomForest using MFCCs + pitch + energy + zero-crossing rate.

In [5]:
import json
import joblib
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.preprocessing import StandardScaler

TEST_ACTORS = {21, 22, 23, 24}  # held-out speakers: 2 male, 2 female

df = pd.read_csv("data/features.csv")
train_df = df[~df["actor"].isin(TEST_ACTORS)].reset_index(drop=True)
test_df = df[df["actor"].isin(TEST_ACTORS)].reset_index(drop=True)
print(f"Train: {len(train_df)} clips ({train_df['actor'].nunique()} actors) | "
      f"Test: {len(test_df)} clips ({test_df['actor'].nunique()} actors)")

# --- trained model: scaled MFCC + pitch + energy features -> RandomForest ---
scaler = StandardScaler().fit(train_df[MODEL_FEATURES])
X_train = scaler.transform(train_df[MODEL_FEATURES])
X_test = scaler.transform(test_df[MODEL_FEATURES])
y_train, y_test = train_df["emotion"], test_df["emotion"]

clf = RandomForestClassifier(n_estimators=300, max_depth=None, random_state=0)
clf.fit(X_train, y_train)
model_preds = clf.predict(X_test)
model_acc = accuracy_score(y_test, model_preds)

# --- heuristic baseline: nearest centroid on pitch + energy only ---
h_scaler = StandardScaler().fit(train_df[HEURISTIC_FEATURES])
Xh_train = h_scaler.transform(train_df[HEURISTIC_FEATURES])
Xh_test = h_scaler.transform(test_df[HEURISTIC_FEATURES])
centroids = {
    label: Xh_train[(train_df["emotion"] == label).values].mean(axis=0)
    for label in sorted(train_df["emotion"].unique())
}


def nearest_centroid_predict(X_scaled, centroids):
    labels = list(centroids.keys())
    centroid_matrix = np.array([centroids[l] for l in labels])
    dists = np.linalg.norm(X_scaled[:, None, :] - centroid_matrix[None, :, :], axis=2)
    return [labels[i] for i in dists.argmin(axis=1)]


heuristic_preds = nearest_centroid_predict(Xh_test, centroids)
heuristic_acc = accuracy_score(y_test, heuristic_preds)

labels = sorted(df["emotion"].unique())
print(f"\nModel accuracy:     {model_acc:.1%}")
print(f"Heuristic accuracy: {heuristic_acc:.1%}")
print("\nModel classification report:\n", classification_report(y_test, model_preds))

Train: 1200 clips (20 actors) | Test: 240 clips (4 actors)

Model accuracy:     54.6%
Heuristic accuracy: 32.5%

Model classification report:
               precision    recall  f1-score   support

       angry       0.45      0.72      0.55        32
        calm       0.66      0.84      0.74        32
     disgust       0.39      0.56      0.46        32
     fearful       0.65      0.41      0.50        32
       happy       0.43      0.50      0.46        32
     neutral       0.80      0.25      0.38        16
         sad       0.46      0.19      0.27        32
   surprised       0.89      0.75      0.81        32

    accuracy                           0.55       240
   macro avg       0.59      0.53      0.52       240
weighted avg       0.58      0.55      0.53       240



In [6]:
os.makedirs("model", exist_ok=True)
joblib.dump({"scaler": scaler, "clf": clf, "features": MODEL_FEATURES}, "model/model.joblib")
joblib.dump(
    {"scaler": h_scaler, "centroids": centroids, "features": HEURISTIC_FEATURES},
    "model/heuristic.joblib",
)

eval_results = {
    "test_actors": sorted(TEST_ACTORS),
    "n_train": len(train_df),
    "n_test": len(test_df),
    "labels": labels,
    "model_accuracy": model_acc,
    "heuristic_accuracy": heuristic_acc,
    "model_confusion_matrix": confusion_matrix(y_test, model_preds, labels=labels).tolist(),
    "heuristic_confusion_matrix": confusion_matrix(y_test, heuristic_preds, labels=labels).tolist(),
    "model_report": classification_report(y_test, model_preds, output_dict=True),
    "heuristic_report": classification_report(y_test, heuristic_preds, output_dict=True),
    "test_predictions": [
        {
            "file": f"data/raw/Actor_{row.actor:02d}/" + Path(row.file).name,
            "actor": int(row.actor),
            "gender": row.gender,
            "true": row.emotion,
            "model_pred": model_preds[i],
            "heuristic_pred": heuristic_preds[i],
        }
        for i, row in enumerate(test_df.itertuples())
    ],
}
with open("model/eval_results.json", "w") as f:
    json.dump(eval_results, f, indent=2)

print("Wrote model/model.joblib, model/heuristic.joblib, model/eval_results.json")

Wrote model/model.joblib, model/heuristic.joblib, model/eval_results.json


## 5. Download the trained artifacts

Zips `model/model.joblib`, `model/heuristic.joblib`, `model/eval_results.json` and downloads
them.

In [7]:
import shutil
from google.colab import files

shutil.make_archive("trained_model", "zip", "model")
files.download("trained_model.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Drop all three into your local project's `model/` folder, then run the below commands in your local:

```bash
cd voice-emotion-recognition
source .venv/bin/activate   # or: python -m venv .venv && pip install -r requirements.txt
streamlit run streamlit_app.py
```